In [70]:
import numpy
import torch
import matplotlib
import PIL
import os
from tqdm import tqdm
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [2]:
labels = os.listdir('D:\\DATA\\Amateur Drawing Semantic Segmentations\\20240716-1034 (1)\\labels')
copy_to = 'D:\\DATA\\Amateur Drawing Semantic Segmentations\\20240716-1034 (1)\\drawings'
os.makedirs(copy_to, exist_ok=True)

In [3]:
drawings_dir = '..\\..\\DATA\\Amateur_Drawings_Dataset_official\\amateur_drawings'

In [26]:
for label in tqdm(labels):
    for set_id in os.listdir(drawings_dir):
        set_path = f"{drawings_dir}\\{set_id}"
        drawings = os.listdir(set_path)
        for d in drawings:
            name = f"{label.split('_')[0]}.png"
            if name == d:
                shutil.copyfile(f"{set_path}\\{name}", f"{copy_to}\\{name}")

In [53]:
# importing the module
import json

# Opening JSON file
with open('../../../DATA/Amateur Drawing Semantic Segmentations/20240716-1034 (1)/label_definition.json') as json_file:
    data = json.load(json_file)

data['label_name_to_id']['CONFLICT']=26
data['label_name_to_id']['Conflicted']=26
data['label_name_to_id']['Unlabeled']=26
print(data['label_name_to_id'])
print()
print(data['label_name_to_color'])
remap = {0:0, 2:1, 3:2, 4:3, 5:4, 13:5, 17:6, 22:7, 23:8, 26:9}
def img_to_label_id(img):
    img = img.reshape(-1,3)
    b = img[:,0]
    g = img[:,1]
    r = img[:,2]
    labels = np.zeros_like(r)
    for classname in data['label_name_to_color']:
        ref_classname = classname + '' # copy precaution
        if classname not in ['Eyebrows','Mouth','Pupils','Other_Facial_Accessory_','Nose','Teeth','Eyes','Tongue','BACKGROUND','Not_Annotated','Unlabeled']:
            ref_classname = 'Unlabeled'
        color = data['label_name_to_color'][classname]
        mask = (r==color[0]) & (g==color[1]) & (b==color[2])
        labels[mask] = remap[int(data['label_name_to_id'][ref_classname])]
    labels = labels.reshape(1024,1024)
    return labels

{'Lower_leg': 1, 'Eyebrows': 2, 'Mouth': 3, 'Pupils': 4, 'Other_Facial_Accessory_': 5, 'Head_Region': 6, 'Hand': 7, 'Lower_Torso': 8, 'Lower_arm': 9, 'Fingers': 10, 'Skirt': 11, 'Other_Head_accessory_': 12, 'Nose': 13, 'Upper_leg': 14, 'Upper_Torso': 15, 'Upper_arm': 16, 'Teeth': 17, 'Ears': 18, 'Hand_accessory': 19, 'Hair': 20, 'Other_Body_accessory': 21, 'Eyes': 22, 'Tongue': 23, 'Feet': 24, 'Neck': 25, 'BACKGROUND': 0, 'CONFLICT': 26, 'Unlabeled': 26, 'Not_Annotated': 0, 'Conflicted': 26}

{'Lower_leg': [177, 25, 217], 'Eyebrows': [68, 121, 228], 'Mouth': [21, 28, 56], 'Pupils': [63, 128, 243], 'Other_Facial_Accessory_': [171, 69, 206], 'Head_Region': [12, 87, 48], 'Hand': [245, 66, 108], 'Lower_Torso': [125, 172, 35], 'Lower_arm': [117, 209, 198], 'Fingers': [149, 218, 110], 'Skirt': [169, 185, 239], 'Other_Head_accessory_': [71, 196, 14], 'Nose': [160, 234, 179], 'Upper_leg': [16, 162, 174], 'Upper_Torso': [247, 181, 62], 'Upper_arm': [187, 108, 168], 'Teeth': [121, 28, 75], 'Ears

In [49]:
label_dir = 'D:\\DATA\\Amateur Drawing Semantic Segmentations\\20240716-1034 (1)\\labels'
drawing_dir = 'D:\\DATA\\Amateur Drawing Semantic Segmentations\\20240716-1034 (1)\\drawings'
os.makedirs(f"{label_dir}_face", exist_ok=True)
os.makedirs(f"{drawing_dir}_face", exist_ok=True)

In [117]:
for label in tqdm(os.listdir(label_dir)[10:]):
    img = cv2.imread(f"{label_dir}\\{label}")
    drawing = cv2.imread(f"{drawing_dir}\\{label.split('_')[0]}.png")
    img = cv2.resize(img, (1024,1024), cv2.INTER_NEAREST)
    drawing = cv2.resize(drawing, (1024,1024), cv2.INTER_NEAREST)
    label_img = img_to_label_id(img)
    mask = (label_img>0) & (label_img<9)
    Xs = np.where(mask>0)[0]
    Ys = np.where(mask>0)[1]
    pad = 40
    try:
        bbox = np.array([min(Ys)-pad,min(Xs)-pad,max(Ys)+pad,max(Xs)+pad])
        img_cropped = label_img[bbox[1]:bbox[3],bbox[0]:bbox[2]]
        drawing_cropped = drawing[bbox[1]:bbox[3],bbox[0]:bbox[2]]
        img_cropped = cv2.resize(img_cropped, (1024,1024), cv2.INTER_NEAREST)
        drawing_cropped = cv2.resize(drawing_cropped, (1024,1024), cv2.INTER_LINEAR)
        cv2.imwrite(f"{label_dir}_face\\{label}",img_cropped)
        cv2.imwrite(f"{drawing_dir}_face\\{label}",drawing_cropped)
    except:
        continue

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 303/303 [00:42<00:00,  7.11it/s]
